# M8: which tracked files make `git_dirty` true on Kaggle?

Every Kaggle run records `git_dirty: true`, and the zips don't say why. This notebook repeats the setup of `m8_kaggle.ipynb` at the same commit (`cb9b1dc`), and records `git status` after each step, using the same command as `src/ids/utils/run.py`. It also tests whether the partition rebuilt on Kaggle is the same, row for row, as the one built on the local PC.

**Attach as input:** the same private Kaggle dataset as `m3_kaggle.ipynb`. **Settings:** Internet *On*, Accelerator *None*. **Time:** about 2 minutes. No model is trained.

**Output:** `/kaggle/working/m8_dirty_check.json`. Download it and bring it back.


In [ ]:
import hashlib, importlib.metadata as md, json, os, subprocess, sys
import pandas as pd
COMMIT = "cb9b1dca37b21603218633d268fccd83bb5d6d93"
REPO = "/kaggle/working/IDS-code-check"
EXPECT = {
 "category_intro/assignments.parquet": [
  9038114,
  "1df03190eb7f3d8d23bf384bc9c67e156b5cbdff9190a6b33291d96fdae1f18b"
 ],
 "category_intro/dpub.parquet": [
  90429,
  "db865144c44877117d5c40eae8fc99740f6c00cdc3bf8f28c47b50fbbd54862f"
 ],
 "category_intro/ble_assignments.parquet": [
  1532922,
  "55d5b45b78333ad662a482fca4844928979938533743deaa55878ca6c152d0d4"
 ],
 "alt_order/assignments.parquet": [
  9038114,
  "15c8b3adb3f50e8f2f6ff683d5032878afdfa12960dcecea9bae14e7ffc02124"
 ],
 "alt_order/dpub.parquet": [
  90429,
  "98b7f63cdd84ee09493035ae4c8b1e69807710d09e9cfdd9fcada3c952d54758"
 ],
 "alt_order/ble_assignments.parquet": [
  1532922,
  "55d5b45b78333ad662a482fca4844928979938533743deaa55878ca6c152d0d4"
 ]
}   # rows and fingerprint of each file, from the local PC's build at the same commit
def sh(*a):
    return subprocess.run(a, cwd=REPO, capture_output=True, text=True)
def status():
    r = sh("git", "status", "--porcelain", "--untracked-files=no")   # the same command as run.py
    return r.stdout.splitlines() if r.returncode == 0 else ["git failed: " + r.stderr]
def fp(path):
    d = pd.read_parquet(path)
    h = hashlib.sha256()
    for c in d.columns:
        h.update(c.encode())
        h.update(d[c].to_numpy().tobytes())
    return [len(d), h.hexdigest()]
out = {"commit": COMMIT, "status": {}}
subprocess.run(["git", "clone", "-q", "https://github.com/Indrajit147/IDS-code.git", REPO], check=True)
subprocess.run(["git", "-C", REPO, "checkout", "-q", COMMIT], check=True)
out["status"]["after_clone"] = status()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", REPO, "--no-deps"], check=True)
out["status"]["after_pip_install"] = status()
os.environ["IDS_REPO_ROOT"] = REPO
m2 = subprocess.run([sys.executable, "scripts/m2_build_partitions.py", "--env", "kaggle", "--pools", "full"], cwd=REPO, capture_output=True, text=True)
out["m2_exit_code"] = m2.returncode
out["m2_log_tail"] = m2.stdout[-1500:] + m2.stderr[-1500:]
out["status"]["after_m2"] = status()
out["numstat"] = sh("git", "diff", "--numstat").stdout.splitlines()
out["text_diff"] = sh("git", "diff", "--", "*.md", "*.json").stdout[:20000]
got = {f: fp(f"/kaggle/working/splits/full/{f}") for f in EXPECT}
out["partition"] = {f: {"rows": got[f][0], "same_as_local": got[f] == EXPECT[f], "fingerprint": got[f][1]} for f in EXPECT}
out["packages"] = {p: md.version(p) for p in ["numpy", "pandas", "pyarrow", "matplotlib"]}
out["python"] = sys.version.split()[0]
json.dump(out, open("/kaggle/working/m8_dirty_check.json", "w"), indent=1)
print(json.dumps({k: v for k, v in out.items() if k not in ("text_diff", "m2_log_tail")}, indent=1))
